# P05 — private E46 Vi-Qwen 3B, one GPU, direct max1536
Uses saved P00 top20, exact E45 top12-v2 selector and E00 source metadata. Loads the completed fresh E46 adapter, generates every private question once from its original prompt at greedy max1536, then applies complete E43+E44 Unified Clean. No reranker, no private reference answers, no max1024 stage. Per-question records support fail-closed resume. This creates one submission ZIP; it does **not** submit it.

Attach code-v95 (this notebook's code package), complete P00, E00-v2, E45, completed E46, and the private questions dataset. A single sufficiently large GPU is required for full FP16.

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, shutil, subprocess, sys
os.environ.setdefault('CUDA_VISIBLE_DEVICES','0')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
INPUT=Path('/kaggle/input')
CONFIG_NAME='final-private-p05-e46-top12v2-max1536-unified-v1.json'
config_hits=list(INPUT.rglob(CONFIG_NAME))
assert len(config_hits)==1, config_hits
CONFIG=config_hits[0]; ROOT=CONFIG.parents[1]
SCRIPT=ROOT/'scripts/run_final_private_p05_e46_kaggle.py'
assert SCRIPT.is_file(), SCRIPT
PRIVATE_HITS=list(INPUT.rglob('private-official.json'))
P00_HITS=[]
for path in INPUT.rglob('report.json'):
    try: report=json.loads(path.read_text(encoding='utf-8'))
    except (OSError,ValueError): continue
    if report.get('experiment_id')=='FINAL-private-p00-retrieval-parent-v1' and (path.parent/'retrieval/candidate-pool-top20.jsonl').is_file(): P00_HITS.append(path.parent)
assert P00_HITS, 'Attach the complete saved P00 output'
matching=[]
for source in P00_HITS:
    report=json.loads((source/'report.json').read_text(encoding='utf-8'))
    for private in PRIVATE_HITS:
        if hashlib.sha256(private.read_bytes()).hexdigest()==report.get('private_questions_sha256'): matching.append((source,private))
assert matching, 'P00 and private questions do not match'
assert len({hashlib.sha256((s/'report.json').read_bytes()).hexdigest() for s,_ in matching})==1, 'Conflicting P00 reports'
P00,PRIVATE=sorted(matching,key=lambda item: str(item[0]))[0]
E46_HITS=[]
for path in INPUT.rglob('adapter-final/complete.json'):
    try: value=json.loads(path.read_text(encoding='utf-8'))
    except (OSError,ValueError): continue
    if value.get('experiment_id')=='E46-viqwen-top12v2-fulltrain7000-v1' and (path.parent.parent/'training-identity.json').is_file(): E46_HITS.append(path.parent.parent)
assert E46_HITS, 'Attach a completed E46 output'
assert len({json.loads((p/'adapter-final/complete.json').read_text(encoding='utf-8'))['adapter_sha256'] for p in E46_HITS})==1, 'Conflicting E46 adapters'
E46=sorted(E46_HITS)[0]
training_identity=json.loads((E46/'training-identity.json').read_text(encoding='utf-8'))
E45_HITS=[]
for path in INPUT.rglob('report.json'):
    if hashlib.sha256(path.read_bytes()).hexdigest()==training_identity['e45_report_sha256'] and (path.parent/'training-data/summary.json').is_file(): E45_HITS.append(path.parent)
assert E45_HITS, 'Attach the E45 output used by this E46 adapter'
E45=sorted(E45_HITS)[0]
metadata=json.loads((ROOT/'configs/e45-top12v2-fulltrain7000-v1.json').read_text(encoding='utf-8'))['metadata_source']
E00_HITS=[p.parent for p in INPUT.rglob(metadata['chunks_path']) if (p.parent/metadata['documents_path']).is_file() and hashlib.sha256(p.read_bytes()).hexdigest()==metadata['chunks_sha256']]
assert E00_HITS, 'Attach the pinned E00-v2 chunks/documents artifact'
E00=sorted(E00_HITS)[0]
OUTPUT=Path('/kaggle/working/final-private-p05-e46-top12v2-max1536-unified-v1')
resume_hits=[]
for path in INPUT.rglob('preflight.json'):
    try: value=json.loads(path.read_text(encoding='utf-8'))
    except (OSError,ValueError): continue
    if value.get('experiment_id')=='FINAL-private-p05-e46-top12v2-max1536-unified-v1': resume_hits.append(path.parent)
if resume_hits and not OUTPUT.exists():
    ranked=sorted(resume_hits,key=lambda p: sum(1 for _ in (p/'generation/records').glob('*.json')),reverse=True)
    if len(ranked)>1 and sum(1 for _ in (ranked[0]/'generation/records').glob('*.json'))==sum(1 for _ in (ranked[1]/'generation/records').glob('*.json')): raise RuntimeError('Ambiguous P05 resume sources; detach one')
    shutil.copytree(ranked[0],OUTPUT); print('Resuming P05:',ranked[0])
OUTPUT.mkdir(parents=True,exist_ok=True)
FLAGS=['--inventory-reviewed','--experiment-plan-reviewed','--private-inference-authorized','--submission-creation-authorized','--e45-selector-reviewed','--e46-adapter-reviewed','--unified-clean-authorized']
BASE=['--project-root',str(ROOT),'--p00-artifact',str(P00),'--e00-artifact',str(E00),'--e45-artifact',str(E45),'--e46-artifact',str(E46),'--private',str(PRIVATE),'--output',str(OUTPUT),'--config',str(CONFIG),*FLAGS]
print({'code':str(ROOT),'p00':str(P00),'e00':str(E00),'e45':str(E45),'e46':str(E46),'private':str(PRIVATE),'output':str(OUTPUT),'gpu':'cuda:0','max_new_tokens':1536})
subprocess.run([sys.executable,'-u',str(SCRIPT),'preflight',*BASE],check=True)

In [ ]:
# Rebuild E45 top12-v2 from P00 top20; never use private answers.
subprocess.run([sys.executable,'-u',str(SCRIPT),'prepare',*BASE],check=True)
print(json.dumps(json.loads((OUTPUT/'plan.json').read_text(encoding='utf-8')),ensure_ascii=False,indent=2))

In [ ]:
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
PIN={'torch':'2.10.0+cu128','transformers':'5.16.1','peft':'0.19.1','accelerate':'1.13.0'}
if version('torch')!=PIN['torch']: raise RuntimeError(f'Need torch {PIN["torch"]}; current={version("torch")}')
needed=[f'{name}=={wanted}' for name,wanted in PIN.items() if name!='torch' and version(name)!=wanted]
if needed: subprocess.run([sys.executable,'-m','pip','install','-q',*needed],check=True)
if version('torchao'): subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()==1, f'Expose one GPU; visible={torch.cuda.device_count()}'
print({'gpu':torch.cuda.get_device_name(0),'vram_gb':round(torch.cuda.get_device_properties(0).total_memory/2**30,2)})
from huggingface_hub import snapshot_download
MODEL_ID='AITeamVN/Vi-Qwen2-3B-RAG'; REVISION='eaf427c24d86066a2b35828c499b7db3af321227'
try: MODEL=Path(snapshot_download(repo_id=MODEL_ID,revision=REVISION,local_files_only=True))
except Exception: MODEL=Path(snapshot_download(repo_id=MODEL_ID,revision=REVISION))
assert MODEL.name==REVISION, MODEL

In [ ]:
# One FP16 model on cuda:0. Every completed QID is durably checkpointed.
subprocess.run([sys.executable,'-u',str(SCRIPT),'generate',*BASE,'--model-cache',str(MODEL)],check=True)

In [ ]:
subprocess.run([sys.executable,'-u',str(SCRIPT),'finalize',*BASE],check=True)
report=json.loads((OUTPUT/'report.json').read_text(encoding='utf-8'))
FINAL=Path('/kaggle/working/submission-p05-e46-top12v2-max1536.zip')
shutil.copy2(OUTPUT/'submission.zip',FINAL)
print(json.dumps({'status':'VALID — PRIVATE E46 READY FOR MANUAL REVIEW','questions':report['sample_size'],'zip':str(FINAL),'eos':report['diagnostics']['eos_questions'],'length':report['diagnostics']['length_questions'],'cleaned':report['diagnostics']['changed_questions'],'reranker':report['selected_stack']['reranker'],'private_reference_answers_read':report['private_reference_answers_read']},ensure_ascii=False,indent=2))
print('Save Version with Always Save Output. Do not submit until you review the report and ZIP.')